In [1]:
import os
nnn = 1
os.environ["OMP_NUM_THREADS"] = str(nnn) # export OMP_NUM_THREADS=1
os.environ["OPENBLAS_NUM_THREADS"] = str(nnn) # export OPENBLAS_NUM_THREADS=1
os.environ["MKL_NUM_THREADS"] = str(nnn) # export MKL_NUM_THREADS=1
os.environ["VECLIB_MAXIMUM_THREADS"] = str(nnn) # export VECLIB_MAXIMUM_THREADS=1
os.environ["NUMEXPR_NUM_THREADS"] = str(nnn)  # export NUMEXPR_NUM_THREADS=1

os.environ["TOKENIZERS_PARALLELISM"] = "false"
import pandas as pd
from pathlib import Path

import shutil
USE_SLURM = False
if shutil.which("squeue"):
    print("Slurm is available on this system.")
    USE_SLURM = True
else:
    print("Slurm is not available.")

Slurm is not available.


In [2]:
from TELF.pipeline.blocks import DataBundle, SAVE_DIR_BUNDLE_KEY, SOURCE_DIR_BUNDLE_KEY
from TELF.pipeline import BlockManager  

from TELF.pipeline.blocks import (
    DataBundle,
    VultureCleanBlock,
    BeaverVocabBlock,
    OrcaBlock,
    WolfBlock,
    CleanDuplicatesBlock,
    CleanAffiliationsBlock,
    BeaverDocWordBlock,
    SemanticHNMFkBlock,
    ArticFoxBlock,
    TermAttributionBlock,
    LoadTermsBlock,
    TermAttributionBlock,
    SBatchBlock,
    ClusteringAnalyzerBlock,
    PeacockStatsBlock,
    PipelineSummaryBlock,
    CollectHNMFkLeafBlock,
    TermiteNeo4jBlock,
    TermiteVectorBlock,
    TermTableBlock,
)

/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/pymilvus/client/__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


# Load Data

In [3]:
df = pd.read_csv(Path("..") / ".." / ".." /"data" / "sample2.csv").head(50)
EXAMPLE_OUTPUT = Path( "example_results") / 'semantic_HNMFk_collection_slurm_option' 
bundle = DataBundle({'Default.df':df, 
                     SAVE_DIR_BUNDLE_KEY: EXAMPLE_OUTPUT,
                     SOURCE_DIR_BUNDLE_KEY: EXAMPLE_OUTPUT})
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   eid               50 non-null     object 
 1   s2id              50 non-null     object 
 2   doi               50 non-null     object 
 3   title             50 non-null     object 
 4   abstract          50 non-null     object 
 5   year              50 non-null     int64  
 6   authors           50 non-null     object 
 7   author_ids        50 non-null     object 
 8   affiliations      50 non-null     object 
 9   funding           5 non-null      object 
 10  PACs              8 non-null      object 
 11  publication_name  50 non-null     object 
 12  subject_areas     50 non-null     object 
 13  s2_authors        50 non-null     object 
 14  s2_author_ids     50 non-null     object 
 15  citations         45 non-null     object 
 16  references        38 non-null     object 
 17 

# Build the Blocks

In [4]:
duplicate_cleaner_block = CleanDuplicatesBlock()
orca_block = OrcaBlock()
clean_affiliations_block = CleanAffiliationsBlock()
vulture_block = VultureCleanBlock(verbose=True, 
                                  use_substitutions=True,
                                  init_settings={"n_jobs":-1, 'parallel_backend': 'threading'})
vocab_block = BeaverVocabBlock(call_settings={'min_df':3, 'max_df':0.6, 'max_features':10000})

[CleanDuplicates] needs → (df)   provides → (df)
[Orca] needs → (df)   provides → (df, map)
[CleanAffiliations] needs → (df)   provides → (df)
[VultureClean] needs → (df, substitutions)   provides → (df, vulture_steps)
[BeaverVocab] needs → (df)   provides → (vocabulary)


In [5]:
terms_block = LoadTermsBlock( call_settings={SOURCE_DIR_BUNDLE_KEY: Path('../../../data/sample_terms3.md')})
out_terms = terms_block(bundle=bundle)
out_terms.substitutions


[Terms] needs → (dir)   provides → (terms, substitutions, substitutions_reverse, query)


{'decision-making': 'decision-making',
 'self-supervised learning': 'self-supervised_learning',
 'neural architecture search': 'neural_architecture_search',
 'edge computing': 'edge_computing',
 'mystery': 'mystery',
 'machine learning': 'machine_learning',
 'malware': 'malware',
 'ransomware': 'ransomware',
 'matrix': 'matrix',
 'anomaly': 'anomaly',
 'anomaly detection': 'anomaly_detection',
 'cluster analysis': 'cluster_analysis',
 'cluster': 'cluster',
 'unsupervised': 'unsupervised'}

In [6]:
matrix_block = BeaverDocWordBlock(tag="DocWord", needs=("df", "vocabulary",))

[DocWord] needs → (df, vocabulary)   provides → (X)


In [7]:
nmfk_params = {
            "n_perturbs": 2,
            "n_iters": 2,
            "epsilon": 0.015,
            "n_jobs": -1,
            "init": "nnsvd",
            "use_gpu": False,
            "save_output": True,
            "collect_output": True,
            "predict_k_method": "sill",
            "verbose": True,
            "nmf_verbose": False,
            "transpose": False,
            "sill_thresh": 0.8,
            "pruned": True,
            "nmf_method": "nmf_fro_mu",
            "calculate_error": True,
            "predict_k": True,
            "use_consensus_stopping": 0,
            "calculate_pac": True,
            "consensus_mat": True,
            "perturb_type": "uniform",
            "perturb_multiprocessing": False,
            "perturb_verbose": False,
            "simple_plot": True,
            "k_search_method": "bst_pre",
            "H_sill_thresh": 0.1,
            "clustering_method": "kmeans",
            "device": -1,
        }

In [8]:
semantic_hfactor_block = SemanticHNMFkBlock(
    needs=("DocWord.X", "df", "vocabulary", ),
    init_settings={
        "depth":2, 
        "sample_thresh":5,
        "Ks_deep_max":30,
        "nmfk_params":[nmfk_params],
    },
    call_settings={
        "Ks":range(2, 10),
    }
)

if USE_SLURM:
    sbatch_hnmfk = SBatchBlock(
        wrapped_block=semantic_hfactor_block,
        venv_type="conda",
        venv_path="TELF2",
    )

    hnmfk_block = sbatch_hnmfk
else:
    hnmfk_block = semantic_hfactor_block



[SemanticHNMFk] needs → (DocWord.X, df, vocabulary)   provides → (model, model_path)


In [9]:
wolf_coauthor_block = WolfBlock(tag="WolfAuthor", category='co-author')
wolf_coaffiliation_block = WolfBlock(tag="WolfAffil", category='co-affiliation')
term_attribute_block =   TermAttributionBlock( )
post_process_block = ArticFoxBlock(call_settings={"ollama_model":"llama3.2:3b-instruct-fp16"})

[WolfAuthor] needs → (df, map)   provides → (graph_co-author)
[WolfAffil] needs → (df, map)   provides → (graph_co-affiliation)
[Attribution] needs → (df, terms)   provides → (df, term_representation_df)
[ArticFox] needs → (df, vocabulary, model_path)   provides → (block_status)


In [10]:
hnmfk_analyzer = ClusteringAnalyzerBlock(
    tag='HNMFAnalyzer',
    mode='hnmf'
)

summary_block = PipelineSummaryBlock()
peacock_block = PeacockStatsBlock()

collect_leaves = CollectHNMFkLeafBlock(  call_settings={"hnmfk_dir": "./example_results/semantic_HNMFk_collection_slurm_option/07_SemanticHNMFk"},)

[HNMFAnalyzer] needs → (df, hnmfk_model, vocabulary)   provides → (clusters_path)
[PipelineSummary] needs → (∅)   provides → (docs_summary_df, docs_summary_plot)
[PeacockStats] needs → (df, save_path)   provides → (outpath)
[LeafDataLabels] needs → (df, save_path)   provides → (leaf_data_csv, leaf_labels_csv)


In [11]:
neo4j_block = TermiteNeo4jBlock(call_settings={
    "raw_csv_path": bundle.get("LeafDataLabels.leaf_data_csv"),
    "neo4j_uri": "neo4j://localhost:7666",
    "neo4j_user": "neo4j",
    "neo4j_pass": "local_password",
})
vector_store_block = TermiteVectorBlock(call_settings={
    "raw_csv_path": bundle.get("LeafDataLabels.leaf_data_csv"),
    "id_column": "eid",
    "text_column": "abstract",
    "index_name": "termite_vectors_test_e2e",
    "model_name": "malteos/scincl",
    "env": {  # optional override; defaults match your script
        "EMBEDDING_STORE": "opensearch",
        "OS_HOST": "localhost",
        "OS_PORT": "9200",
        "OS_USE_SSL": "false",
    },
    # Optional smoke test:
    # "test_query_text": "What problem in real-world malware labeling does the HNMFk Classifier aim to solve?",
    # "test_k": 5,
})

[TermiteNeo4j] needs → (leaf_data_csv, leaf_labels_csv)   provides → (data_triplets_csv, topic_triplets_csv)
[TermiteVectorIndex] needs → (leaf_data_csv)   provides → (vector_index_name, vector_stats)


# Block Manager

In [ ]:
manager = BlockManager(
    blocks = [
        duplicate_cleaner_block,
        vulture_block,
        orca_block,
        clean_affiliations_block,
        vocab_block,
        matrix_block,
        hnmfk_block,
        term_attribute_block,
        wolf_coauthor_block,
        wolf_coaffiliation_block,  
        post_process_block,   
        peacock_block,
        collect_leaves,
        summary_block,
        neo4j_block,
        vector_store_block
    ],
    databundle=bundle,  
    progress   = True,          # see which block is executing
    capture_output=None #'file',
)

Block (tag)                                │ Needs (✓/✗)                    │ Provides
──────────────────────────────────────────────────────────────────────────────────────
VultureCleanBlock (VultureClean)           │ df, substitutions              │ ['df', 'vulture_steps']
OrcaBlock (Orca)                           │ df                             │ ['df', 'map']
CleanAffiliationsBlock (CleanAffiliations) │ df                             │ ['df']
BeaverVocabBlock (BeaverVocab)             │ df                             │ ['vocabulary']
BeaverDocWordBlock (DocWord)               │ df, vocabulary                 │ ['X']
SemanticHNMFkBlock (SemanticHNMFk)         │ DocWord.X, df, vocabulary      │ ['model', 'model_path']
TermAttributionBlock (Attribution)         │ df, terms                      │ ['df', 'term_representation_df']
WolfBlock (WolfAuthor)                     │ df, map                        │ ['graph_co-author']
WolfBlock (WolfAffil)                      │ df, map       

In [13]:
bundle = manager()

▶  [1/15] 01_VultureClean …
[01_VultureClean] ✔ loaded from checkpoint
✓  [1/15] 01_VultureClean finished in 2.03s
▶  [2/15] 02_Orca …
[02_Orca] ✔ loaded from checkpoint
✓  [2/15] 02_Orca finished in 0.01s
▶  [3/15] 03_CleanAffiliations …
✓  [3/15] 03_CleanAffiliations finished in 0.00s
▶  [4/15] 04_BeaverVocab …
✓  [4/15] 04_BeaverVocab finished in 0.01s
▶  [5/15] 05_DocWord …
✓  [5/15] 05_DocWord finished in 0.01s
▶  [6/15] 06_SemanticHNMFk …
Continuing from checkpoint...
Loading saved object state from checkpoint...
Done
Loading saved object state from checkpoint...
[06_SemanticHNMFk] ⭳ checkpoint saved
✓  [6/15] 06_SemanticHNMFk finished in 0.01s
▶  [7/15] 07_Attribution …
[07_Attribution] ✔ loaded from checkpoint
✓  [7/15] 07_Attribution finished in 0.01s
▶  [8/15] 08_WolfAuthor …
[08_WolfAuthor] ✔ loaded from checkpoint
✓  [8/15] 08_WolfAuthor finished in 0.00s
▶  [9/15] 09_WolfAffil …
[09_WolfAffil] ✔ loaded from checkpoint
✓  [9/15] 09_WolfAffil finished in 0.00s
▶  [10/15] 10_

100%|██████████| 42/42 [00:00<00:00, 2101.76it/s]


{'entity_type': 'Topic_ID', 'unique': True, 'from_column': 'Graph_Name'}
entity_map[FROM_COL] =Graph_Name
entity={'entity': 'Root_0_0', 'weight': None, 'attributes': None} for entity_map[ET] =Topic_ID
{'entity_type': 'Document_ID', 'from_column': 'doi', 'attribute_columns': [{'from_column': 'title', 'attribute_name': 'Title'}, {'from_column': 'eid', 'attribute_name': 'EID'}, {'from_column': 's2id', 'attribute_name': 'S2ID'}, {'from_column': 'doi', 'attribute_name': 'DOI'}], 'unique': True}
entity_map[FROM_COL] =doi
entity={'entity': '8b37f74e-ec68-44fe-88a9-5830cdf7ea48', 'weight': None, 'attributes': [('Title', 'AI-Driven Forecasting Models in Finance'), ('EID', '4b230c89-61aa-4b61-8780-87c12fbf9183'), ('S2ID', '43245345-6e05-4a65-96fe-9b8bd15bb9ad'), ('DOI', '8b37f74e-ec68-44fe-88a9-5830cdf7ea48')]} for entity_map[ET] =Document_ID
{'entity_type': 'Year', 'from_column': 'year', 'attribute_columns': None, 'attribute_function': None, 'unique': True}
entity_map[FROM_COL] =year
entity={'e

  0%|          | 0/9 [00:00<?, ?it/s]
Traceback (most recent call last):
  File "/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/TELF/pipeline/block_manager.py", line 147, in __call__
    self.bundle = block(self.bundle)
                  ^^^^^^^^^^^^^^^^^^
  File "/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/TELF/pipeline/blocks/base_block.py", line 166, in __call__
    self.run(bundle)
  File "/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/TELF/pipeline/blocks/termite_neo4j_block.py", line 402, in run
    termite.from_csv_to_triplets(str(topic_csv_path), str(topic_triplets_path), topic_triplet_map)
  File "/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/TELF/applications/Termite/termite.py", line 120, in from_csv_to_triplets
    self.graph_injector.from_csv_to_triplets(csv_path, save_path, column_triplet_map)
  File "/Users/barron/anaconda3/envs/TELF/lib/python3.11/site-packages/TELF/applications/Termite/neo4j_termite/Da

{'entity_type': 'Topic_ID', 'unique': True, 'from_column': 'Graph_Name', 'attribute_columns': [{'from_column': 'label', 'attribute_name': 'label'}, {'from_column': 'Graph_Name', 'attribute_name': 'Graph_Name'}]}
entity_map[FROM_COL] =Graph_Name
entity={'entity': 'Root_0_0', 'weight': None, 'attributes': [('label', 'Robust algorithms for classification tasks.'), ('Graph_Name', 'Root_0_0')]} for entity_map[ET] =Topic_ID
row_entities={'Topic_ID': {'entity': 'Root_0_0', 'weight': None, 'attributes': [('label', 'Robust algorithms for classification tasks.'), ('Graph_Name', 'Root_0_0')]}}
⚠️ Exception in block 14_TermiteNeo4j:


AttributeError: 'Series' object has no attribute 'parent_name'

In [ ]:
bundle.keys()